## Cell type based coloring

In [ ]:
## The below snippets of code will have to be run for individual sample separately.

In [1]:
library(ggplot2)
library(glue)
library(R.matlab)
library(stringr)
library(dplyr)

all_sample_names <- c("SMC01-N", "SMC01-T", "SMC02-N", "SMC02-T", "SMC03-N", "SMC03-T", "SMC04-N", "SMC04-T", "SMC05-N", "SMC05-T", "SMC06-N", "SMC06-T",
                      "SMC07-N", "SMC07-T", "SMC08-N", "SMC08-T", "SMC09-N", "SMC09-T", "SMC10-N", "SMC10-T", "SMC11-T", "SMC14-T", "SMC15-T", "SMC16-T",
                      "SMC17-T", "SMC18-T", "SMC19-T", "SMC20-T", "SMC21-T", "SMC22-T", "SMC23-T", "SMC24-T", "SMC25-T")

df <- read.csv("../../pipeline_testing/pareto_front_consolidated_data.csv", row.names=1)

## Here the sample for which you want to plot has to be choosen
sample <- all_sample_names[33]
sample_df <- df[df$Sample == sample,]

# extract archetypes coordinates
parti_output_path <- paste("../parti_result", sample, sep="/")
files <- list.files(parti_output_path)        
parti_output_file <- files[str_detect(files, ".mat$")]

parti_output <- readMat(paste(parti_output_path, parti_output_file, sep="/"))

arc <- as.data.frame(parti_output$arc)
arc <- arc[,1:3]
colnames(arc) <- c("PC1", "PC2", "PC3")

## Path where all the plots and its associated data will be saved
export_path <- "../../pipeline_testing/pareto_front_plots_test"    
dir.create(paste(export_path, sample, sep="/"), recursive = TRUE)

write.csv(arc, glue("../../pipeline_testing/pareto_front_plots_test/{sample}/arc_coordinates.csv"))

pdf(file = glue("../../pipeline_testing/pareto_front_plots_test/{sample}/{sample}_2d.pdf"),height=20,width=30)
plot_cell_type <- ggplot(sample_df, aes(x=PC1,y=PC2,color=Cell_type),size=20)+
                          geom_point(size = 6)+
                          geom_point(data=arc,aes(x=PC1,y=PC2),color='black',size=15)+
                          guides(color = guide_legend(override.aes = list(size = 20)))+
                          ggtitle(sample)+
                          labs(x="PC1",y="PC2", color="Cell Types")+
                          theme_classic()+
                          theme(
                            axis.line = element_line(color = "black", linewidth = 2.5),
                            axis.ticks = element_line(color = "black", linewidth = 2),
                            axis.text = element_text(color = "black", size = 80),
                            axis.ticks.length = unit(0.2, "cm")
                          )
  
plot_cell_type + scale_color_manual(values=c("T cells" = "Blue", "Myeloids" = "deeppink", "Epithelial cells" = "darkgreen", "Stromal cells" = "darkmagenta"))+  
  theme(text = element_text(size = 95)) + annotate("text", x = arc[,1]+3, y = arc[,2]+3, label = rownames(arc),size=32, fontface="bold")

dev.off()

R.matlab v3.7.0 (2022-08-25 21:52:34 UTC) successfully loaded. See ?R.matlab for help.


Attaching package: ‘R.matlab’


The following objects are masked from ‘package:base’:

    getOption, isOpen



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message in fortify(data, ...):
“Arguments in `...` must be used.
✖ Problematic argument:
• size = 20
ℹ Did you misspell an argument name?”


agg_record_416818b35707 
                      2

In [2]:
library(scatterplot3d)

my_cols <- c("T cells" = "Blue", "Myeloids" = "deeppink", "Epithelial cells" = "darkgreen", "Stromal cells" = "darkmagenta")
sample_df$color_vector <- my_cols[sample_df$Cell_type]

pdf(file = glue("../../pipeline_testing/pareto_front_plots_test/{sample}/{sample}_3d.pdf"),height=20,width=30)
par(mar = c(6, 10, 7, 8))

x_lims <- c(min(sample_df$PC1) - 5, max(sample_df$PC1) + 5)
y_lims <- c(min(sample_df$PC2) - 5, max(sample_df$PC2) + 5)
z_lims <- c(min(sample_df$PC3) - 5, max(sample_df$PC3) + 5)

s3d <- scatterplot3d(sample_df$PC1, sample_df$PC2, sample_df$PC3, 
                     color = sample_df$color_vector, 
                     pch = 19,            
                     cex.symbols = 2.7,     
                     cex.axis = 3,       
                     cex.lab = 4,        
                     cex.main = 5.5,      
                     xlim = x_lims, ylim = y_lims, zlim = z_lims,
                     main = sample,
                     xlab = "PC1",
                     ylab = "PC2",
                     zlab = "",
                     angle = 45,
                     grid = TRUE, box = FALSE)
mtext("PC3", side = 2, line = 0, cex = 4, font = 1)

s3d$points3d(arc$PC1, arc$PC2, arc$PC3, 
             col = "black", 
             pch = 16,   
             cex = 4)   

text_coords <- s3d$xyz.convert(arc$PC1 + 0.5, arc$PC2 + 0.5, arc$PC3 + 0.5)

text(text_coords$x, text_coords$y, 
     labels = rownames(arc), 
     pos = 3,            
     cex = 4,         
     col = "black", 
     font = 2) 

par(xpd = TRUE)
legend("topright", 
       legend = names(my_cols), 
       col = my_cols, 
       pch = 16, 
       inset = c(-0.01, 0), 
       cex = 3.5,           
       title = "Cell Types",
       bty = "n")

dev.off()

agg_record_41683550e2b6 
                      2

## Mellon density inside pareto front

In [ ]:
## The below snippets of code will have to be run for individual sample separately.

In [3]:
all_sample_names <- c("SMC01-N", "SMC01-T", "SMC02-N", "SMC02-T", "SMC03-N", "SMC03-T", "SMC04-N", "SMC04-T", "SMC05-N", "SMC05-T", "SMC06-N", "SMC06-T",
                     "SMC07-N", "SMC07-T", "SMC08-N", "SMC08-T", "SMC09-N", "SMC09-T", "SMC10-N", "SMC10-T", "SMC11-T", "SMC14-T", "SMC15-T", "SMC16-T",
                     "SMC17-T", "SMC18-T", "SMC19-T", "SMC20-T", "SMC21-T", "SMC22-T", "SMC23-T", "SMC24-T", "SMC25-T")

df <- read.csv("../../pipeline_testing/pareto_front_consolidated_data.csv", row.names=1)
sample_df <- df[df$Sample == sample,]

In [4]:
library(glue)
arc <-  read.csv(glue("../../pipeline_testing/pareto_front_plots/{sample}/arc_coordinates.csv"), row.names=1)

In [5]:
pdf(file = glue("../../pipeline_testing/pareto_front_plots_test/{sample}/{sample}_2d_mellon_density.pdf"),height=20,width=30)
plot_cell_type <- ggplot(sample_df, aes(x=PC1,y=PC2,color=mellon_log_density),size=20)+
                          geom_point(size = 8, alpha = 1)+
                          geom_point(data=arc,aes(x=PC1,y=PC2),color='black',size=15)+
                          scale_color_distiller(
                                palette = "Spectral", 
                                # direction = -1,
                                name = "Mellon Density",
                                guide = guide_colourbar(
                                  barwidth = 4, 
                                  barheight = 60,
                                  title.position = "top",      
                                  title.hjust = 0.5,
                                )
                          )+
                          ggtitle(sample)+
                          # labs(x="PC1",y="PC2", color="Mellon Density")+
                          theme_classic()+
                          theme(
                            axis.line = element_line(color = "black", linewidth = 2.5),
                            axis.ticks = element_line(color = "black", linewidth = 2),
                            axis.text = element_text(color = "black", size = 80),
                            axis.ticks.length = unit(0.2, "cm"),
                            legend.title = element_text(size = 60, face = "bold"), 
                            legend.title.align = 0.5
                          )
  
plot_cell_type + theme(text = element_text(size = 95)) + annotate("text", x = arc[,1]+3, y = arc[,2]+3, label = rownames(arc),size=32, fontface="bold")

dev.off()

Warning message in fortify(data, ...):
“Arguments in `...` must be used.
✖ Problematic argument:
• size = 20
ℹ Did you misspell an argument name?”


agg_record_4168420415f3 
                      2

## Cells coloring based on functional diversification inside pareto front

In [ ]:
## The below snippets of code will be run for all samples together and plot the functional identity associated with different specialists
## across all samples

In [10]:
library(viridis)
library(viridisLite)
library(pals)

plot_data_order_epi <- read.csv("../../pipeline_testing/some_extra_plots/epithelial_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_myo <- read.csv("../../pipeline_testing/some_extra_plots/myeloid_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_t <- read.csv("../../pipeline_testing/some_extra_plots/T_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_stromal <- read.csv("../../pipeline_testing/some_extra_plots/stromal_archetype_functional_identity_gavish.csv", row.names=1)
all_pathway <- c(plot_data_order_epi$pathway, plot_data_order_myo$pathway, plot_data_order_t$pathway, plot_data_order_stromal$pathway)
all_pathway <- unique(all_pathway)

all_possible_groups <- all_pathway
my_pal <- colorRampPalette(polychrome())(length(all_possible_groups))
names(my_pal) <- all_possible_groups

In [11]:
library(glue)
library(stringr)
library(dplyr)
library(ggplot2)
library(R.matlab)

samples <- c("SMC01-N", "SMC01-T", "SMC02-T", "SMC03-N", "SMC03-T", "SMC04-N", "SMC04-T", "SMC05-N", "SMC05-T", "SMC06-N", "SMC06-T",
             "SMC07-N", "SMC07-T", "SMC08-N", "SMC08-T", "SMC09-N", "SMC09-T", "SMC10-N", "SMC10-T", "SMC11-T", "SMC14-T", "SMC15-T", "SMC16-T",
             "SMC17-T", "SMC18-T", "SMC19-T", "SMC20-T", "SMC21-T", "SMC22-T", "SMC23-T", "SMC24-T", "SMC25-T")

for (i in 1:length(samples)){
    sample <- samples[i]
    
    main_dir <- glue("../../pipeline_testing/mellon_analysis/{sample}")
    
    target_files <- list.files(path = main_dir, pattern = "_specialists_and_generalists.*\\.csv$", full.names = TRUE)
    
    sg_data_list <- lapply(target_files, function(file){
                        read.csv(file, row.names=1)
                    })
    
    filenames <- tools::file_path_sans_ext(basename(target_files))
    arc_names <- ifelse(grepl("generalists_cell", filenames, ignore.case = TRUE), "Generalist", gsub("_specialists.*", "", filenames))
    
    sg_data_list <- Map(function(df, n) {
      df$archetype <- n
      return(df)
    }, sg_data_list, arc_names)
    
    sg_data_merged <- do.call(rbind, sg_data_list)
    
    rownames(sg_data_merged) <- NULL
    
    common_cell_data <- readLines(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"))
    
    if (length(common_cell_data)>0){
        cell_removed_before_mellon <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"), sep="\t")
        names(cell_removed_before_mellon) <- 'cells'
        cell_removed_before_mellon$archetype <- NA
        sg_data_merged <- rbind(sg_data_merged, cell_removed_before_mellon)
    }
    epi_func_ident <- read.csv("../../pipeline_testing/some_extra_plots/epithelial_archetype_functional_identity_gavish.csv", row.names=1)
    myo_func_ident <- read.csv("../../pipeline_testing/some_extra_plots/myeloid_archetype_functional_identity_gavish.csv", row.names=1)
    tnk_func_ident <- read.csv("../../pipeline_testing/some_extra_plots/T_archetype_functional_identity_gavish.csv", row.names=1)
    strommal_func_ident <- read.csv("../../pipeline_testing/some_extra_plots/stromal_archetype_functional_identity_gavish.csv", row.names=1)
    
    func_ident_list <- list(epi_func_ident, myo_func_ident, tnk_func_ident, strommal_func_ident)
    merged_func_ident <- Reduce(function(x, y) merge(x, y, all = TRUE), func_ident_list)
    sample_func_ident <- merged_func_ident[merged_func_ident$sample_name == sample,]
    sample_func_ident <- sample_func_ident %>%
                              group_by(sublist_names) %>%
                              slice_max(order_by = percentage, n = 1, with_ties = FALSE) %>%
                              ungroup()
    
    sample_func_ident <- sample_func_ident %>% 
                             rename(archetype = sublist_names)
    
    sample_func_ident <- sample_func_ident %>%
                              mutate(archetype = str_replace_all(archetype, " ", "_")) %>%
                              mutate(archetype = str_replace_all(archetype, "generalists_cell", "Generalist"))

    parti_output <- readMat(glue("../../pipeline_testing/parti_result/{sample}/Variables.mat"))
    pc_data <- parti_output$pc

    cells <- read.table(glue("../../pipeline_testing/data/{sample}/{sample}_required_cell_modified.tsv"), sep="\t", header=T)
    rownames(pc_data) <- cells$index
    pc_data <- pc_data[,1:3]
    colnames(pc_data) <- c('pc1', 'pc2', 'pc3')
    merged_final_data <- merge(pc_data, sg_data_merged, by.x = "row.names", by.y = "cells", all = TRUE)

    merged_final_data <- merged_final_data %>%
                          left_join(sample_func_ident %>% select(archetype,pathway),by = "archetype")
                                
    export_path <- "../../pipeline_testing/pareto_front_plots_test"
    dir.create(paste(export_path, sample, sep="/"), recursive = TRUE)
                                
    write.csv(merged_final_data, glue("../../pipeline_testing/pareto_front_plots_test/{sample}/functional_identity_pareto_front.csv"))

    merged_final_data_mod <- merged_final_data[!is.na(merged_final_data$archetype), ]

    # pdf(glue("../../pipeline_testing/pareto_front_plots/{sample}/Functional_identity_specialist_generalist_2D.pdf"),width=25, height=16)
    pdf(glue("../../pipeline_testing/pareto_front_plots_test/{sample}/Functional_identity_specialist_generalist_2D_without_transitory.pdf"),width=25, height=16)
    p <- ggplot(merged_final_data_mod, aes(x = pc1, y = pc2, color = pathway)) +
          geom_point(size = 4, alpha = 0.8) +  
          scale_color_manual(values = my_pal) +
          labs(
            title = "Functional diversification in 2D pareto space",
            x = "PC1",
            y = "PC2",
            color = "Functional identity"
          ) +
          theme_classic(base_size = 24) +    
          theme(
            plot.title = element_text(size = 48, face = "bold", hjust = 0.5),
            axis.text.x = element_text(size = 38, color = "black"),
            axis.text.y = element_text(size = 38, color = "black"),
            axis.title = element_text(size = 44),
            legend.title = element_text(size = 36, face = "italic"),
            legend.text = element_text(size = 30),
            plot.title.position = "plot"
          )+
          guides(fill = guide_legend(ncol = 1))

    print(p)
    
    dev.off()
}

Warning message in dir.create(paste(export_path, sample, sep = "/"), recursive = TRUE):
“'../../pipeline_testing/pareto_front_plots_test/SMC01-N' already exists”
Warning message in dir.create(paste(export_path, sample, sep = "/"), recursive = TRUE):
“'../../pipeline_testing/pareto_front_plots_test/SMC01-T' already exists”
